## Wstęp do Uczenia Maszynowego - Projekt I
### Etap: Kamień Milowy III
### Autorzy: Krzysztof Osiński, Jakub Miszczak

### Import packages and data:

In [ ]:
import pandas as pd
import numpy as np
import sklearn 
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
np.random.seed(23)
import zipfile

zip_path = "fraud-detection-transactions-dataset.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open("synthetic_fraud_dataset.csv") as file:
        df = pd.read_csv(file)

df.columns = df.columns.str.replace(" ","_").str.lower()
df1 = df.drop(['transaction_id','user_id'],axis='columns')
df1['timestamp'] = pd.to_datetime(df1['timestamp'])

X = df1.drop('fraud_label', axis='columns')
Y = df1['fraud_label']

from sklearn.preprocessing import MinMaxScaler

scaled_columns = X.select_dtypes(['int64', 'float64']).columns

scaler = MinMaxScaler()

X[scaled_columns] = scaler.fit_transform(X[scaled_columns])

### Przypomnienie wyników z kamienia milowego II:

W trakcie wszystkich testów przeprowadzonych podczas kamienia milowego II wyszło nam, że interesujące nas kolumny wybierzemy wsród kolumn: failed_transaction_count_7d, high_failed_tx_flag, sqrt_risk_score, risk_score, high_sqrt_risk_score, failed_tx_risk_interaction, amount_risk_interaction.

Musieliśmy wybrać czy zdecydować się na normalny risk_score czy też ten spierwiastkowany i zdecydowaliśmy się na zwykły, gdyż wśród róznych testów wypadał on zdecydowanie stabilniej niż sqrt_risk_score.

In [ ]:
# X['high_failed_tx_flag'] = (X['failed_transaction_count_7d'] > 3).astype(int)    //nw czy w koncu to chciałeś używać

X['failed_tx_risk_interaction'] = X['failed_transaction_count_7d'] * X['risk_score']
X['failed_tx_ratio'] = X['failed_transaction_count_7d'] / (X['daily_transaction_count'] + 1)
X['amount_risk_interaction'] = X['transaction_amount'] * X['risk_score']

### Tworzenie Modelu i sprawdzanie go:

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt



df1['failed_tx_risk_interaction'] = df1['failed_transaction_count_7d'] * df1['risk_score']
df1['failed_tx_ratio'] = df1['failed_transaction_count_7d'] / (df1['failed_transaction_count_7d'] + df1['daily_transaction_count'])

selected_features = [
    'failed_tx_ratio',
    'failed_tx_risk_interaction',
    'failed_transaction_count_7d',
    'risk_score',  # <-- zostawiamy oryginalny
    'amount_risk_interaction'
]

X_selected = X[selected_features]

#X_selected = pd.concat([X_selected, X_encoded_df], axis=1)

print(f'Finalny kształt danych: {X_selected.shape}')
X_selected.head()


# 1. Podział danych
X_train, X_test, y_train, y_test = train_test_split(X_selected, Y, test_size=0.2, stratify=Y, random_state=42)

# 2. Modele do przetestowania
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Random_Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
}

print("=== Cross-validation scores (F1) ===")
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(f"{name}: F1 = {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# 3. Dostrajanie hiperparametrów dla najlepszego modelu (np. Random Forest)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

# 4. Trening i ewaluacja wszystkich modeli
print("\n=== Wyniki dla wszystkich modeli ===")
for name, model in models.items():
    model.fit(X_train, y_train)  # Trening modelu
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n--- {name} ---")
    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred))

    # 5. Confusion Matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()